# 05. 6-Way Multilingual Translation Instruction Generator
Demonstrates `InstructionTaskGenerator` and `LexicalTaskGenerator` on a sample of the corpus (full-scale generation is run via `scripts/generate_tasks.sh`, ~187K tasks for the full training split).

In [ ]:
# ============================================================
# PATH BOOSTER — Guarantees project root in sys.path & CWD
# ============================================================
import os, sys
try:
    cwd = os.getcwd()
except FileNotFoundError:
    cwd = os.path.expanduser('~')
    os.chdir(cwd)
proj_dir = os.path.join(os.path.expanduser('~'), 'Ekegusii-LLM-Translation-main')
if os.path.isdir(proj_dir):
    os.chdir(proj_dir)
elif os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')
if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())


In [1]:
import os

if "COLAB_GPU" in os.environ or os.environ.get("COLAB_RELEASE_TAG"):
    if not os.path.exists("Ekegusii-LLM-Translation"):
        os.system("git clone https://github.com/aykahsay/Ekegusii-LLM-Translation.git")
    os.chdir("Ekegusii-LLM-Translation")
    os.system("pip install -q -r requirements.txt")
elif not os.path.exists("src") and os.path.basename(os.getcwd()) == "notebooks":
    # Running locally via `jupyter nbconvert` from within notebooks/ --
    # the repo root (containing src/, data/) is one directory up.
    os.chdir("..")

import sys
sys.path.insert(0, os.getcwd())

import logging
logging.basicConfig(level=logging.INFO, format="%(levelname)s | %(message)s")


In [2]:
import os, sys
p = os.path.join(os.path.expanduser('~'), 'Ekegusii-LLM-Translation-main')
if os.path.isdir(p) and p not in sys.path: sys.path.insert(0, p); os.chdir(p)

from src.master_corpus.manager import MasterCorpusManager
from src.task_generation.instruction_generator import InstructionTaskGenerator
from src.task_generation.lexical_tasks import LexicalTaskGenerator

manager = MasterCorpusManager()
sample = manager.load_train_split().sample(200, random_state=42)

INFO | Loaded dataset split [master_train.csv]: 39,421 rows.


## Sentence-level 6-way tasks

In [3]:
sentence_gen = InstructionTaskGenerator(manager)
tasks_df = sentence_gen.generate_tasks_from_dataframe(sample)
print(f'{len(sample)} concepts -> {len(tasks_df)} instruction tasks')
tasks_df['task_type'].value_counts()

INFO | Expanded 200 concepts into 952 instruction tasks.


200 concepts -> 952 instruction tasks


task_type
ENG_to_EKE    188
EKE_to_ENG    188
SWA_to_ENG    150
ENG_to_SWA    150
EKE_to_SWA    138
SWA_to_EKE    138
Name: count, dtype: int64

In [4]:
tasks_df.iloc[0][['task_type', 'prompt', 'response']]

task_type                                           ENG_to_EKE
prompt       Translate the following English text into Ekeg...
response     Ninkorusie nkorusie ase amaboko ’abakori amabe...
Name: 0, dtype: object

## Lexical-corpus tasks

In [5]:
lexical_gen = LexicalTaskGenerator(manager)
lexical_tasks_df = lexical_gen.generate_all_tasks()
print(f'{len(lexical_tasks_df)} lexical tasks generated')
lexical_tasks_df['task_type'].value_counts()

INFO | Loaded Master Lexical Corpus: 268 terms.


INFO | Generated 536 lexical instruction tasks from 268 lexicon entries.


536 lexical tasks generated


task_type
SWA_to_EKE_lexical    268
EKE_to_SWA_lexical    268
Name: count, dtype: int64